# نوت‌بوک ۶ — ساخت نمودارها برای مقاله

این نوت‌بوک پنج نمودار اصلی مقاله را از فایل‌های JSON نتایج می‌سازد. هر نمودار در دو فرمت `.png` و `.pdf` ذخیره می‌شود.

**لیبل‌ها انگلیسی هستند** — این انتخاب آگاهانه است: matplotlib بدون پکیج‌های اضافی نمی‌تواند فارسی را به‌درستی رندر کند. لیبل‌های انگلیسی روی نمودار + توضیح فارسی در زیرنویس مقاله، رویه استاندارد ژورنال‌های فارسی است.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import json
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/persian-ai-research')
FIGURES_DIR = DRIVE_ROOT / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    'font.size': 11, 'font.family': 'DejaVu Sans',
    'axes.titlesize': 12, 'axes.labelsize': 11,
    'savefig.dpi': 300, 'savefig.bbox': 'tight',
})

with open(DRIVE_ROOT / 'results' / 'baseline_results.json', encoding='utf-8') as f:
    baseline = json.load(f)
with open(DRIVE_ROOT / 'results' / 'error_analysis.json', encoding='utf-8') as f:
    err = json.load(f)
with open(DRIVE_ROOT / 'results' / 'train_on_one_matrix.json', encoding='utf-8') as f:
    train_one = json.load(f)
with open(DRIVE_ROOT / 'results' / 'train_on_two_matrix.json', encoding='utf-8') as f:
    train_two = json.load(f)
with open(DRIVE_ROOT / 'results' / 'robustness_results.json', encoding='utf-8') as f:
    robustness = json.load(f)
print('✓ همه نتایج بارگذاری شد.')

## نمودار ۱ — ماتریس تعمیم‌پذیری بین‌مولدی

In [ ]:
models_short = ['DeepSeek', 'GPT-4o-mini', 'Gemini']
models_full = ['deepseek-v3.1', 'gpt-4o-mini', 'gemini-2.5-flash-lite']

rows = []; row_labels = []
rows.append([err['by_model'][m]['accuracy'] for m in models_full])
row_labels.append('All 3 (baseline)')

for m_short, key in zip(models_short, ['deepseek', 'gpt', 'gemini']):
    rows.append([train_one['matrix'][f'trained_on_{key}']['by_test_generator'][mm] for mm in models_full])
    row_labels.append(f'Only {m_short}')

label_map = {
    'train_deepseek_gpt': ('DeepSeek+GPT', 'gemini-2.5-flash-lite'),
    'train_deepseek_gemini': ('DeepSeek+Gemini', 'gpt-4o-mini'),
    'train_gpt_gemini': ('GPT+Gemini', 'deepseek-v3.1'),
}
for key, (label, held) in label_map.items():
    r = train_two['results'][key]
    row = [r['acc_on_seen'].get(mm, r['acc_on_unseen']) for mm in models_full]
    rows.append(row)
    row_labels.append(label)

matrix = np.array(rows)

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(matrix, cmap='RdYlGn', vmin=0.85, vmax=1.0)
ax.set_xticks(range(3)); ax.set_xticklabels(models_short)
ax.set_yticks(range(len(row_labels))); ax.set_yticklabels(row_labels)
ax.set_xlabel('Test on (generator)', fontweight='bold')
ax.set_ylabel('Training condition', fontweight='bold')
ax.set_title('Cross-Generator Generalization Accuracy', fontweight='bold')
for i in range(matrix.shape[0]):
    for j in range(matrix.shape[1]):
        ax.text(j, i, f'{matrix[i,j]:.3f}', ha='center', va='center',
                color='white' if matrix[i,j] < 0.92 else 'black', fontweight='bold')
ax.axhline(y=0.5, color='black', linewidth=2)
ax.axhline(y=3.5, color='black', linewidth=1, linestyle='--', alpha=0.5)
plt.colorbar(im, ax=ax, shrink=0.8, label='Accuracy')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig1_generalization_matrix.png')
plt.savefig(FIGURES_DIR / 'fig1_generalization_matrix.pdf')
plt.show()

## نمودار ۲ — مقاومت در برابر بازنویسی

In [ ]:
cats = ['Overall', 'Human\n(unchanged)', 'AI: DeepSeek', 'AI: GPT-4o-mini', 'AI: Gemini']
before = [
    baseline['test_metrics']['eval_accuracy'],
    err['by_model']['human']['accuracy'],
    err['by_model']['deepseek-v3.1']['accuracy'],
    err['by_model']['gpt-4o-mini']['accuracy'],
    err['by_model']['gemini-2.5-flash-lite']['accuracy'],
]
after = [
    robustness['paraphrased_accuracy'],
    robustness['human_accuracy'],
    robustness['by_model']['deepseek-v3.1']['accuracy'],
    robustness['by_model']['gpt-4o-mini']['accuracy'],
    robustness['by_model']['gemini-2.5-flash-lite']['accuracy'],
]
x = np.arange(len(cats)); w = 0.35
fig, ax = plt.subplots(figsize=(10, 6))
ax.bar(x - w/2, before, w, label='Original text', color='#2E86AB')
ax.bar(x + w/2, after, w, label='Back-translated text', color='#E63946')
for i, (b, a) in enumerate(zip(before, after)):
    ax.text(i - w/2, b + 0.01, f'{b:.3f}', ha='center', fontsize=9, fontweight='bold')
    ax.text(i + w/2, a + 0.01, f'{a:.3f}', ha='center', fontsize=9, fontweight='bold')
ax.axhline(y=0.5, color='gray', linestyle='--', linewidth=1)
ax.set_xticks(x); ax.set_xticklabels(cats)
ax.set_ylabel('Accuracy', fontweight='bold')
ax.set_title('Detector Robustness Under Back-Translation Paraphrasing', fontweight='bold')
ax.set_ylim(0, 1.1); ax.legend(loc='lower left'); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig2_robustness_paraphrasing.png')
plt.savefig(FIGURES_DIR / 'fig2_robustness_paraphrasing.pdf')
plt.show()

## نمودار ۳ — تحلیل خطا (سه پنل)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

# پنل الف
ax = axes[0]
mods = ['human', 'deepseek-v3.1', 'gpt-4o-mini', 'gemini-2.5-flash-lite']
labels = ['Human', 'DeepSeek', 'GPT-4o', 'Gemini']
accs = [err['by_model'][m]['accuracy'] for m in mods]
ax.bar(labels, accs, color=['#264653','#2A9D8F','#E9C46A','#F4A261'])
for i, a in enumerate(accs):
    ax.text(i, a + 0.005, f'{a:.3f}', ha='center', fontweight='bold')
ax.set_ylim(0.88, 1.02); ax.set_ylabel('Accuracy', fontweight='bold')
ax.set_title('(a) Accuracy by source', fontweight='bold')
ax.grid(axis='y', alpha=0.3)

# پنل ب
ax = axes[1]
cat_map = {'علوم نظری':'Theoretical','علوم مهندسی':'Engineering','علوم پزشکی':'Medical','علوم انسانی':'Humanities','تاریخ و زندگی‌نامه':'History/Bio'}
cats = list(err['by_category'].keys())
cat_lbls = [cat_map.get(c, c) for c in cats]
cat_accs = [err['by_category'][c]['accuracy'] for c in cats]
ax.bar(cat_lbls, cat_accs, color='#457B9D')
for i, a in enumerate(cat_accs):
    ax.text(i, a + 0.003, f'{a:.3f}', ha='center', fontweight='bold')
ax.set_ylim(0.93, 1.02); ax.set_title('(b) Accuracy by domain', fontweight='bold')
ax.tick_params(axis='x', rotation=20); ax.grid(axis='y', alpha=0.3)

# پنل ج
ax = axes[2]
buckets = list(err['by_length'].keys())
blbls = ['40-50','50-80','80-120','120+']
baccs = [err['by_length'][b]['accuracy'] for b in buckets]
bcounts = [err['by_length'][b]['count'] for b in buckets]
ax.bar(blbls, baccs, color='#9D4EDD')
for i, (a, c) in enumerate(zip(baccs, bcounts)):
    ax.text(i, a + 0.003, f'{a:.3f}\n(n={c})', ha='center', fontsize=8, fontweight='bold')
ax.set_ylim(0.93, 1.02); ax.set_xlabel('Text length (words)', fontweight='bold')
ax.set_title('(c) Accuracy by length', fontweight='bold'); ax.grid(axis='y', alpha=0.3)

plt.suptitle('Error Analysis on Baseline Detector', fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig3_error_analysis.png')
plt.savefig(FIGURES_DIR / 'fig3_error_analysis.pdf')
plt.show()

## نمودار ۴ — ماتریس درهم‌ریختگی پیش/پس از حمله

In [ ]:
cm_before = np.array([[186, 14], [4, 596]])
cm_after_d = robustness['confusion_matrix_after']
cm_after = np.array([[cm_after_d['human_predicted_human'], cm_after_d['human_predicted_ai']],
                     [cm_after_d['ai_predicted_human'], cm_after_d['ai_predicted_ai']]])

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
vmax = max(cm_before.max(), cm_after.max())
for ax, cm, title in zip(axes, [cm_before, cm_after], ['Before paraphrasing\n(baseline)', 'After paraphrasing\n(back-translation)']):
    ax.imshow(cm, cmap='Blues', vmin=0, vmax=vmax)
    ax.set_xticks([0,1]); ax.set_yticks([0,1])
    ax.set_xticklabels(['Human','AI']); ax.set_yticklabels(['Human','AI'])
    ax.set_xlabel('Predicted', fontweight='bold'); ax.set_ylabel('True', fontweight='bold')
    ax.set_title(title, fontweight='bold')
    for i in range(2):
        for j in range(2):
            ax.text(j, i, str(cm[i,j]), ha='center', va='center',
                    color='white' if cm[i,j] > vmax*0.5 else 'black', fontweight='bold', fontsize=14)
plt.suptitle('Confusion Matrix: Impact of Paraphrasing Attack', fontweight='bold', y=1.05)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig4_confusion_matrix.png')
plt.savefig(FIGURES_DIR / 'fig4_confusion_matrix.pdf')
plt.show()

## نمودار ۵ — خلاصه عملکرد در شرایط مختلف

In [ ]:
conditions = ['Baseline\n(in-dist.)', 'Train on 1\nmodel only', 'Train on 2\n(LOO avg)', 'After\nparaphrasing']
baseline_acc = baseline['test_metrics']['eval_accuracy']
train_one_avg = np.mean([v['overall'] for v in train_one['matrix'].values()])
train_two_avg = np.mean([v['acc_overall'] for v in train_two['results'].values()])
para_acc = robustness['paraphrased_accuracy']
values = [baseline_acc, train_one_avg, train_two_avg, para_acc]
colors = ['#2A9D8F','#E9C46A','#F4A261','#E63946']

fig, ax = plt.subplots(figsize=(9, 5.5))
ax.bar(conditions, values, color=colors, width=0.6)
for i, v in enumerate(values):
    ax.text(i, v + 0.015, f'{v:.3f}', ha='center', fontweight='bold')
ax.axhline(y=0.5, color='gray', linestyle='--', linewidth=1)
ax.text(0.02, 0.52, 'Random baseline', ha='left', fontsize=9, color='gray',
        transform=ax.get_yaxis_transform())
ax.annotate(f'−{baseline_acc - para_acc:.3f}',
            xy=(3, para_acc), xytext=(3, para_acc + 0.15),
            fontsize=11, color='#E63946', fontweight='bold', ha='center',
            arrowprops=dict(arrowstyle='->', color='#E63946', lw=1.5))
ax.set_ylabel('Accuracy', fontweight='bold')
ax.set_title('Performance Across Experimental Conditions', fontweight='bold')
ax.set_ylim(0, 1.1); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig5_performance_summary.png')
plt.savefig(FIGURES_DIR / 'fig5_performance_summary.pdf')
plt.show()
print('\n✓ هر پنج نمودار در figures/ ذخیره شد.')